In [1]:
# Setup and imports
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import cv2
import pickle
import importlib
import os
import datetime

# Import orig_clases - vehicle classes we want to avoid
import consts
importlib.reload(consts)
orig_clases = consts.orig_clases

from aruco_pose import get_camera_angles_from_frame
from capture_utils_v2 import CaptureSystem

tt = lambda x: torch.tensor(cv2.cvtColor(x, cv2.COLOR_BGR2RGB)/255.).permute(2,0,1).float()

print(f"Original vehicle classes to avoid: {orig_clases.tolist()}")
%matplotlib inline

c:\Users\danny\anaconda3\envs\py310\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Camera Matrix:
[[1.04739831e+03 0.00000000e+00 3.24036999e+02]
 [0.00000000e+00 9.50491793e+02 3.23897832e+02]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]
Distortion Coefficients: [ 2.60138074e-01 -4.37871352e+00  3.56370583e-02 -2.58297180e-03
  2.21970709e+01]
Original vehicle classes to avoid: [817, 705, 609, 586, 436, 627, 468, 621, 803, 407, 408, 751, 717, 866, 661, 864]


In [2]:
# Initialize capture system
system = CaptureSystem()
cap = system.cap
print("✓ Camera initialized")

Pixel format set to RGB8
IC4 Grabber and Sink initialized.
✓ Camera initialized


In [3]:
# Load calibration state
with open("capture_system_state.pkl", "rb") as f:
    system.orig_proj_corners, system.corners_img_proj, system.orig_img = pickle.load(f)
print("✓ Calibration state loaded")

✓ Calibration state loaded


In [4]:
# Load all classifiers for evaluation
print("Loading classifiers...")

# Ensemble v1 classifier (Inception, ResNet, VGG, ViT, DINOv2)
import classfier_ensemble as ensemble_v1
ensemble_v1_weights = ensemble_v1.weights
print("✓ Ensemble v1 loaded (Inception, ResNet, VGG, ViT, DINOv2)")

# Ensemble v2 classifier (ConvNeXt, EfficientNet, MobileNet, Swin)
import classfier_ensemble_v2 as ensemble_v2
ensemble_v2_weights = ensemble_v2.weights
print("✓ Ensemble v2 loaded (ConvNeXt, EfficientNet, MobileNet, Swin)")

# DINOv2 standalone classifier
import classfier_dino as dino
dino_weights = dino.weights
print("✓ DINOv2 standalone classifier loaded")

categories = ensemble_v1_weights.meta["categories"]
print(f"\n✓ All classifiers ready for evaluation")

Loading classifiers...
Loading ensemble models...


Using cache found in C:\Users\danny/.cache\torch\hub\facebookresearch_dinov2_main
A matching Triton is not available, some optimizations will not be enabled
Traceback (most recent call last):
  File "c:\Users\danny\anaconda3\envs\py310\lib\site-packages\xformers\__init__.py", line 57, in _is_triton_available
    import triton  # noqa
ModuleNotFoundError: No module named 'triton'
C:\Users\danny/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
C:\Users\danny/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:27: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
C:\Users\danny/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:33: UserWarning: xFormers is available (Block)
  warnings.warn("xFormers is available (Block)")


✓ All models loaded successfully!
  - Inception V3
  - ResNet18
  - VGG16
  - ViT-B/16
  - DINOv2

✓ Ensemble classifier ready!
  Main function: predict_raw(image)
  Alternatives: predict_raw_weighted(image, weights_dict)
               predict_raw_per_model(image)
               ensemble_predict(image)
✓ Ensemble v1 loaded (Inception, ResNet, VGG, ViT, DINOv2)
Loading ensemble v2 models...
✓ All models loaded successfully!
  - ConvNeXt Base
  - EfficientNet B0
  - MobileNetV3 Large
  - Swin Transformer Base

✓ Ensemble classifier v2 ready!
  Main function: predict_raw(image)
  Alternatives: predict_raw_weighted(image, weights_dict)
               predict_raw_per_model(image)
               ensemble_predict(image)
✓ Ensemble v2 loaded (ConvNeXt, EfficientNet, MobileNet, Swin)


Using cache found in C:\Users\danny/.cache\torch\hub\facebookresearch_dinov2_main


✓ DINOv2 standalone classifier loaded

✓ All classifiers ready for evaluation


## Manual Capture Function

This function shows the camera feed and waits for 's' to save frames.

In [5]:
def manual_capture(cap, num_poses=4, window_name="Manual Capture"):
    """
    Manually capture frames by pressing 's'.
    
    Args:
        cap: OpenCV VideoCapture object
        num_poses: Number of poses to capture (default 4)
        window_name: Name for the display window
    
    Returns:
        List of captured frames with pose info
    """
    captured_frames = []
    
    print(f"\n{'='*60}")
    print(f"MANUAL CAPTURE MODE")
    print(f"{'='*60}")
    print(f"Capture {num_poses} poses:")
    print(f"  - Press 's' to SAVE current frame")
    print(f"  - Press 'q' to QUIT early")
    print(f"{'='*60}\n")
    
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(window_name, 1280, 720)
    
    while len(captured_frames) < num_poses:
        ret, frame = cap.read()
        if not ret:
            print("Error: Could not read frame")
            break
        
        # Get ArUco pose
        pose_result = get_camera_angles_from_frame(frame)
        
        # Create display frame with overlay
        display_frame = frame.copy()
        
        # Status bar at top
        cv2.rectangle(display_frame, (0, 0), (display_frame.shape[1], 80), (40, 40, 40), -1)
        
        # Capture count
        status_text = f"Captured: {len(captured_frames)}/{num_poses} | Press 's' to save, 'q' to quit"
        cv2.putText(display_frame, status_text, (10, 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        
        # ArUco pose info
        if pose_result['found']:
            angle = pose_result['angle']
            dist = pose_result['distance_m']
            pose_text = f"Angle: {angle:.1f} deg | Distance: {dist:.2f}m"
            cv2.putText(display_frame, pose_text, (10, 60), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        else:
            cv2.putText(display_frame, "ArUco NOT DETECTED - move camera", (10, 60), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
        
        # Show saved poses info
        if captured_frames:
            y_offset = 100
            for i, cap_info in enumerate(captured_frames):
                saved_text = f"Pose {i+1}: Angle={cap_info['angle']:.1f}° Dist={cap_info['distance']:.2f}m"
                cv2.putText(display_frame, saved_text, (10, y_offset + i*25), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 200, 0), 1)
        
        cv2.imshow(window_name, display_frame)
        
        key = cv2.waitKey(1) & 0xFF
        
        if key == ord('s'):
            if pose_result['found']:
                captured_frames.append({
                    'frame': frame.copy(),
                    'angle': pose_result['angle'],
                    'distance': pose_result['distance_m']
                })
                print(f"✓ Captured pose {len(captured_frames)}/{num_poses}: "
                      f"Angle={pose_result['angle']:.1f}°, Dist={pose_result['distance_m']:.2f}m")
            else:
                print("✗ Cannot save - ArUco marker not detected!")
        
        elif key == ord('q'):
            print("\nCapture cancelled by user")
            break
    
    cv2.destroyWindow(window_name)
    print(f"\n✓ Capture complete: {len(captured_frames)} frames saved")
    return captured_frames

## Experiment 1: CAPAA Approach

Load and project the CAPAA adversarial pattern, then capture 4 poses.

In [ ]:
# Load CAPAA image
from omegaconf import DictConfig
import sys
sys.path.append(r'C:\git\CAPAA\src\python')
from utils import init_prj_window

# CAPAA pattern path
capaa_img_path = r"C:\git\CAPAA\src\python\data\setups\jeep8\prj\adv\CAPAA_PCNet_l1+ssim_500_24_2000\camdE_caml2\5\classifier_all\img_0001.png"
capaa_img = cv2.cvtColor(cv2.imread(capaa_img_path), cv2.COLOR_BGR2RGB)

# Setup projector
setup_info = DictConfig(dict(
    prj_screen_sz      = (1920, 1080),
    prj_im_sz          = (256, 256),
    prj_offset         = (2560, 0),
    prj_brightness     = 0.5,
    delay_frames = 5,
    delay_time = 0.1,
))

prj = init_prj_window(*setup_info['prj_screen_sz'], 0.5, setup_info['prj_offset'])
prj.set_data(capaa_img)
print("✓ CAPAA pattern projected")

In [ ]:
# Capture 4 poses for CAPAA
print("\n" + "="*60)
print("CAPTURING CAPAA EXPERIMENT")
print("="*60)
capaa_captures = manual_capture(cap, num_poses=4, window_name="CAPAA Capture")

## Experiment 2: Best Patch Approach

Load and project the Best Patch pattern, then capture 4 poses (try to match CAPAA poses).

In [ ]:
# Show target poses from CAPAA to help match them
print("\n" + "="*60)
print("TARGET POSES (from CAPAA captures):")
print("="*60)
for i, cap_info in enumerate(capaa_captures):
    print(f"  Pose {i+1}: Angle={cap_info['angle']:.1f}° Distance={cap_info['distance']:.2f}m")
print("\nTry to capture at similar angles and distances!")

In [6]:
# Load Best Patch image
best_patch_img_path = r'C:\git\PhysicalAdverserialProj\results\best_patch_ensamble_classifier_16x16_16_2025-12-24_10_08.png'
best_patch_img = cv2.imread(best_patch_img_path)
best_patch_img = cv2.cvtColor(best_patch_img, cv2.COLOR_BGR2RGB)

# Project using capture system's method
system.plot_on_screen(best_patch_img)
print("✓ Best Patch pattern projected")

✓ Best Patch pattern projected


In [8]:
# Capture 4 poses for Best Patch
print("\n" + "="*60)
print("CAPTURING BEST PATCH EXPERIMENT")
print("="*60)
print("\nRemember target poses from CAPAA:")
# for i, cap_info in enumerate(capaa_captures):
#     print(f"  Pose {i+1}: Angle={cap_info['angle']:.1f}° Distance={cap_info['distance']:.2f}m")
# print()

best_patch_captures = manual_capture(cap, num_poses=4, window_name="Best Patch Capture")


CAPTURING BEST PATCH EXPERIMENT

Remember target poses from CAPAA:

MANUAL CAPTURE MODE
Capture 4 poses:
  - Press 's' to SAVE current frame
  - Press 'q' to QUIT early

✓ Captured pose 1/4: Angle=11.3°, Dist=1.34m
✓ Captured pose 2/4: Angle=25.8°, Dist=1.29m
✓ Captured pose 3/4: Angle=16.0°, Dist=1.38m
✓ Captured pose 4/4: Angle=9.4°, Dist=1.46m

✓ Capture complete: 4 frames saved


## Compare Poses

Visualize how well the poses match between experiments.

In [ ]:
# Compare captured poses
print("\n" + "="*60)
print("POSE COMPARISON")
print("="*60)
print(f"{'Pose':<6} {'CAPAA Angle':<15} {'BP Angle':<15} {'Diff':<10} | {'CAPAA Dist':<12} {'BP Dist':<12} {'Diff':<10}")
print("-"*90)

for i in range(min(len(capaa_captures), len(best_patch_captures))):
    ca = capaa_captures[i]['angle']
    ba = best_patch_captures[i]['angle']
    cd = capaa_captures[i]['distance']
    bd = best_patch_captures[i]['distance']
    
    angle_diff = abs(ca - ba)
    dist_diff = abs(cd - bd)
    
    print(f"{i+1:<6} {ca:>10.1f}°{'':4} {ba:>10.1f}°{'':4} {angle_diff:>6.1f}° | "
          f"{cd:>10.2f}m{'':1} {bd:>10.2f}m{'':1} {dist_diff:>6.3f}m")

In [ ]:
# Visualize pose comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

capaa_angles = [c['angle'] for c in capaa_captures]
capaa_dists = [c['distance'] for c in capaa_captures]
bp_angles = [c['angle'] for c in best_patch_captures]
bp_dists = [c['distance'] for c in best_patch_captures]

# Angle comparison
ax = axes[0]
x = np.arange(len(capaa_angles))
width = 0.35
ax.bar(x - width/2, capaa_angles, width, label='CAPAA', color='steelblue')
ax.bar(x + width/2, bp_angles, width, label='Best Patch', color='darkorange')
ax.set_xlabel('Pose Number')
ax.set_ylabel('Angle (degrees)')
ax.set_title('Angle Comparison')
ax.set_xticks(x)
ax.set_xticklabels([f'Pose {i+1}' for i in range(len(capaa_angles))])
ax.legend()

# Distance comparison
ax = axes[1]
ax.bar(x - width/2, capaa_dists, width, label='CAPAA', color='steelblue')
ax.bar(x + width/2, bp_dists, width, label='Best Patch', color='darkorange')
ax.set_xlabel('Pose Number')
ax.set_ylabel('Distance (m)')
ax.set_title('Distance Comparison')
ax.set_xticks(x)
ax.set_xticklabels([f'Pose {i+1}' for i in range(len(capaa_angles))])
ax.legend()

# Scatter plot
ax = axes[2]
ax.scatter(capaa_angles, capaa_dists, s=100, marker='o', label='CAPAA', color='steelblue')
ax.scatter(bp_angles, bp_dists, s=100, marker='s', label='Best Patch', color='darkorange')
for i in range(len(capaa_angles)):
    ax.annotate(f'{i+1}', (capaa_angles[i], capaa_dists[i]), fontsize=10, ha='center', va='bottom')
    ax.annotate(f'{i+1}', (bp_angles[i], bp_dists[i]), fontsize=10, ha='center', va='bottom')
ax.set_xlabel('Angle (degrees)')
ax.set_ylabel('Distance (m)')
ax.set_title('Pose Space')
ax.legend()

plt.tight_layout()
plt.savefig('manual_capture_poses.png', dpi=150)
plt.show()
print("✓ Pose comparison saved to 'manual_capture_poses.png'")

## Classify Captured Frames

Run all classifiers on the manually captured frames.

In [ ]:
# Classify all captured frames
results_list = []

for pose_idx in range(min(len(capaa_captures), len(best_patch_captures))):
    capaa_frame = capaa_captures[pose_idx]['frame']
    bp_frame = best_patch_captures[pose_idx]['frame']
    
    # Pose info
    capaa_angle = capaa_captures[pose_idx]['angle']
    capaa_dist = capaa_captures[pose_idx]['distance']
    bp_angle = best_patch_captures[pose_idx]['angle']
    bp_dist = best_patch_captures[pose_idx]['distance']
    
    # Prepare tensors
    capaa_tensor = tt(capaa_frame).unsqueeze(0).cuda()
    bp_tensor = tt(bp_frame).unsqueeze(0).cuda()
    
    result_row = {
        'pose_idx': pose_idx + 1,
        'capaa_angle': capaa_angle,
        'capaa_distance': capaa_dist,
        'bp_angle': bp_angle,
        'bp_distance': bp_dist,
        'angle_diff': abs(capaa_angle - bp_angle),
        'dist_diff': abs(capaa_dist - bp_dist),
    }
    
    with torch.no_grad():
        # Get predictions
        capaa_v1 = ensemble_v1.predict_raw_per_model(capaa_tensor)
        capaa_v2 = ensemble_v2.predict_raw_per_model(capaa_tensor)
        capaa_dino = dino.predict_raw(capaa_tensor)
        
        bp_v1 = ensemble_v1.predict_raw_per_model(bp_tensor)
        bp_v2 = ensemble_v2.predict_raw_per_model(bp_tensor)
        bp_dino = dino.predict_raw(bp_tensor)
    
    # Process V1 models
    for model_name in ['inception', 'resnet', 'vgg', 'vit', 'dino']:
        if model_name in capaa_v1:
            # CAPAA
            probs = capaa_v1[model_name]
            pred_idx = probs[0].argmax().item()
            is_success = pred_idx not in orig_clases.tolist()
            result_row[f'capaa_v1_{model_name}_success'] = is_success
            result_row[f'capaa_v1_{model_name}_class'] = categories[pred_idx]
            
            # Best Patch
            probs = bp_v1[model_name]
            pred_idx = probs[0].argmax().item()
            is_success = pred_idx not in orig_clases.tolist()
            result_row[f'bp_v1_{model_name}_success'] = is_success
            result_row[f'bp_v1_{model_name}_class'] = categories[pred_idx]
    
    # V1 Combined
    if 'ensemble' in capaa_v1:
        pred_idx = capaa_v1['ensemble'][0].argmax().item()
        result_row['capaa_v1_combined_success'] = pred_idx not in orig_clases.tolist()
        result_row['capaa_v1_combined_class'] = categories[pred_idx]
        
        pred_idx = bp_v1['ensemble'][0].argmax().item()
        result_row['bp_v1_combined_success'] = pred_idx not in orig_clases.tolist()
        result_row['bp_v1_combined_class'] = categories[pred_idx]
    
    # Process V2 models
    for model_name in ['convnext', 'efficientnet', 'mobilenet', 'swin']:
        if model_name in capaa_v2:
            # CAPAA
            probs = capaa_v2[model_name]
            pred_idx = probs[0].argmax().item()
            is_success = pred_idx not in orig_clases.tolist()
            result_row[f'capaa_v2_{model_name}_success'] = is_success
            result_row[f'capaa_v2_{model_name}_class'] = categories[pred_idx]
            
            # Best Patch
            probs = bp_v2[model_name]
            pred_idx = probs[0].argmax().item()
            is_success = pred_idx not in orig_clases.tolist()
            result_row[f'bp_v2_{model_name}_success'] = is_success
            result_row[f'bp_v2_{model_name}_class'] = categories[pred_idx]
    
    # V2 Combined
    if 'ensemble' in capaa_v2:
        pred_idx = capaa_v2['ensemble'][0].argmax().item()
        result_row['capaa_v2_combined_success'] = pred_idx not in orig_clases.tolist()
        result_row['capaa_v2_combined_class'] = categories[pred_idx]
        
        pred_idx = bp_v2['ensemble'][0].argmax().item()
        result_row['bp_v2_combined_success'] = pred_idx not in orig_clases.tolist()
        result_row['bp_v2_combined_class'] = categories[pred_idx]
    
    # DINOv2 standalone
    pred_idx = capaa_dino[0].argmax().item()
    result_row['capaa_dino_success'] = pred_idx not in orig_clases.tolist()
    result_row['capaa_dino_class'] = categories[pred_idx]
    
    pred_idx = bp_dino[0].argmax().item()
    result_row['bp_dino_success'] = pred_idx not in orig_clases.tolist()
    result_row['bp_dino_class'] = categories[pred_idx]
    
    results_list.append(result_row)

df = pd.DataFrame(results_list)
print(f"✓ Classified {len(df)} pose pairs")

## Results Tables

In [ ]:
# Generate comparison tables
v1_models = ['inception', 'resnet', 'vgg', 'vit', 'dino', 'combined']
v1_display = ['Inception V3', 'ResNet18', 'VGG16', 'ViT-B/16', 'DINOv2 (V1)', 'COMBINED']

print("\n" + "="*80)
print("TABLE 1: V1 MODELS COMPARISON (Per Pose)")
print("="*80)
print(f"{'Model':<20} {'Pose 1':^12} {'Pose 2':^12} {'Pose 3':^12} {'Pose 4':^12}")
print("-"*80)

for model, display in zip(v1_models, v1_display):
    capaa_col = f'capaa_v1_{model}_success'
    bp_col = f'bp_v1_{model}_success'
    if capaa_col in df.columns:
        row = f"{display:<20}"
        for i in range(len(df)):
            c_win = "✓" if df.iloc[i][capaa_col] else "✗"
            b_win = "✓" if df.iloc[i][bp_col] else "✗"
            row += f" C:{c_win} B:{b_win}  "
        print(row)

In [ ]:
# V2 Models table
v2_models = ['convnext', 'efficientnet', 'mobilenet', 'swin', 'combined']
v2_display = ['ConvNeXt Base', 'EfficientNet B0', 'MobileNetV3', 'Swin Transformer', 'COMBINED']

print("\n" + "="*80)
print("TABLE 2: V2 MODELS COMPARISON (Per Pose)")
print("="*80)
print(f"{'Model':<20} {'Pose 1':^12} {'Pose 2':^12} {'Pose 3':^12} {'Pose 4':^12}")
print("-"*80)

for model, display in zip(v2_models, v2_display):
    capaa_col = f'capaa_v2_{model}_success'
    bp_col = f'bp_v2_{model}_success'
    if capaa_col in df.columns:
        row = f"{display:<20}"
        for i in range(len(df)):
            c_win = "✓" if df.iloc[i][capaa_col] else "✗"
            b_win = "✓" if df.iloc[i][bp_col] else "✗"
            row += f" C:{c_win} B:{b_win}  "
        print(row)

In [ ]:
# DINOv2 standalone table
print("\n" + "="*80)
print("TABLE 3: DINOv2 STANDALONE COMPARISON (Per Pose)")
print("="*80)
print(f"{'Model':<20} {'Pose 1':^12} {'Pose 2':^12} {'Pose 3':^12} {'Pose 4':^12}")
print("-"*80)

row = f"{'DINOv2':<20}"
for i in range(len(df)):
    c_win = "✓" if df.iloc[i]['capaa_dino_success'] else "✗"
    b_win = "✓" if df.iloc[i]['bp_dino_success'] else "✗"
    row += f" C:{c_win} B:{b_win}  "
print(row)

In [ ]:
# Summary table with success rates
print("\n" + "="*80)
print("SUMMARY: SUCCESS RATES")
print("="*80)
print(f"{'Model':<25} {'CAPAA':>10} {'Best Patch':>12} {'Winner':>10}")
print("-"*60)

all_models = [
    ('V1: Inception V3', 'capaa_v1_inception_success', 'bp_v1_inception_success'),
    ('V1: ResNet18', 'capaa_v1_resnet_success', 'bp_v1_resnet_success'),
    ('V1: VGG16', 'capaa_v1_vgg_success', 'bp_v1_vgg_success'),
    ('V1: ViT-B/16', 'capaa_v1_vit_success', 'bp_v1_vit_success'),
    ('V1: DINOv2', 'capaa_v1_dino_success', 'bp_v1_dino_success'),
    ('V1: COMBINED', 'capaa_v1_combined_success', 'bp_v1_combined_success'),
    ('V2: ConvNeXt Base', 'capaa_v2_convnext_success', 'bp_v2_convnext_success'),
    ('V2: EfficientNet B0', 'capaa_v2_efficientnet_success', 'bp_v2_efficientnet_success'),
    ('V2: MobileNetV3', 'capaa_v2_mobilenet_success', 'bp_v2_mobilenet_success'),
    ('V2: Swin Transformer', 'capaa_v2_swin_success', 'bp_v2_swin_success'),
    ('V2: COMBINED', 'capaa_v2_combined_success', 'bp_v2_combined_success'),
    ('DINOv2 Standalone', 'capaa_dino_success', 'bp_dino_success'),
]

summary_data = []
for name, capaa_col, bp_col in all_models:
    if capaa_col in df.columns:
        capaa_rate = df[capaa_col].mean() * 100
        bp_rate = df[bp_col].mean() * 100
        
        if capaa_rate > bp_rate:
            winner = 'CAPAA'
        elif bp_rate > capaa_rate:
            winner = 'Best Patch'
        else:
            winner = 'Tie'
        
        print(f"{name:<25} {capaa_rate:>9.1f}% {bp_rate:>11.1f}% {winner:>10}")
        summary_data.append({'Model': name, 'CAPAA': capaa_rate, 'Best Patch': bp_rate, 'Winner': winner})

summary_df = pd.DataFrame(summary_data)

In [ ]:
# Visualize captured frames side by side
num_poses = min(len(capaa_captures), len(best_patch_captures))
fig, axes = plt.subplots(2, num_poses, figsize=(4*num_poses, 8))

for i in range(num_poses):
    # CAPAA frame
    ax = axes[0, i] if num_poses > 1 else axes[0]
    frame_rgb = cv2.cvtColor(capaa_captures[i]['frame'], cv2.COLOR_BGR2RGB)
    ax.imshow(frame_rgb)
    ax.set_title(f'CAPAA Pose {i+1}\nAngle: {capaa_captures[i]["angle"]:.1f}° Dist: {capaa_captures[i]["distance"]:.2f}m')
    ax.axis('off')
    
    # Best Patch frame
    ax = axes[1, i] if num_poses > 1 else axes[1]
    frame_rgb = cv2.cvtColor(best_patch_captures[i]['frame'], cv2.COLOR_BGR2RGB)
    ax.imshow(frame_rgb)
    ax.set_title(f'Best Patch Pose {i+1}\nAngle: {best_patch_captures[i]["angle"]:.1f}° Dist: {best_patch_captures[i]["distance"]:.2f}m')
    ax.axis('off')

plt.tight_layout()
plt.savefig('manual_capture_frames.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Frame comparison saved to 'manual_capture_frames.png'")

In [ ]:
# Save results
timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H_%M")
output_dir = f"manual_capture_results_{timestamp}"
os.makedirs(output_dir, exist_ok=True)

# Save dataframe
df.to_csv(f"{output_dir}/classification_results.csv", index=False)

# Save summary
summary_df.to_csv(f"{output_dir}/summary.csv", index=False)

# Save captures
with open(f"{output_dir}/capaa_captures.pkl", "wb") as f:
    pickle.dump(capaa_captures, f)
with open(f"{output_dir}/best_patch_captures.pkl", "wb") as f:
    pickle.dump(best_patch_captures, f)

print(f"✓ Results saved to '{output_dir}/'")
print(f"  - classification_results.csv")
print(f"  - summary.csv")
print(f"  - capaa_captures.pkl")
print(f"  - best_patch_captures.pkl")

In [ ]:
# Final summary
print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)
capaa_wins = (summary_df['Winner'] == 'CAPAA').sum()
bp_wins = (summary_df['Winner'] == 'Best Patch').sum()
ties = (summary_df['Winner'] == 'Tie').sum()

print(f"\nModels where CAPAA wins: {capaa_wins}")
print(f"Models where Best Patch wins: {bp_wins}")
print(f"Ties: {ties}")

# Overall average
capaa_cols = [c for c in df.columns if c.startswith('capaa_') and c.endswith('_success')]
bp_cols = [c for c in df.columns if c.startswith('bp_') and c.endswith('_success')]

avg_capaa = df[capaa_cols].mean().mean() * 100
avg_bp = df[bp_cols].mean().mean() * 100

print(f"\nOverall average success rate:")
print(f"  CAPAA: {avg_capaa:.1f}%")
print(f"  Best Patch: {avg_bp:.1f}%")
print(f"  Difference: {avg_capaa - avg_bp:+.1f}%")